In [1]:
from doc_processor import ParserConfig, run_parser
from doc_processor.types import RelevanceMode

from pathlib import Path
from os import listdir

from langfuse import get_client
from langfuse._client.resource_manager import LangfuseResourceManager

get_client().shutdown()
LangfuseResourceManager.reset()

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


In [2]:
doc_dir = Path("doc_samples") / "표준계약서모음(hwp-hwpx)"
doc_out = Path("results")

# Change this index or point `target` directly at a file while testing.
target = doc_dir / sorted(listdir(doc_dir))[2]

target

PosixPath('doc_samples/표준계약서모음(hwp-hwpx)/03. 출판권 설정 계약서.hwp')

In [3]:
print(f"running: {target}")

config = ParserConfig(
    relevance_mode=RelevanceMode.KEYWORD_ONLY,
    boundary_review_enabled=True,
    label_review_enabled=True,
    langfuse_enabled=True,
    max_concurrent_workers=10
)

res = run_parser(target, config=config)

running: doc_samples/표준계약서모음(hwp-hwpx)/03. 출판권 설정 계약서.hwp
2026-04-14 16:27:11,144 | INFO | structure analysis run start source=doc_samples/표준계약서모음(hwp-hwpx)/03. 출판권 설정 계약서.hwp
2026-04-14 16:27:11,206 | INFO | [structure_analysis.load_document] start


2026-04-14 16:27:11,726 | INFO | [structure_analysis.load_document] done goto=screen_relevance
2026-04-14 16:27:11,735 | INFO | [structure_analysis.screen_relevance] start
2026-04-14 16:27:11,737 | INFO | [structure_analysis.screen_relevance] done goto=regex_analysis
2026-04-14 16:27:11,738 | INFO | [structure_analysis.regex_analysis] start
2026-04-14 16:27:11,741 | INFO | [structure_analysis.regex_analysis] done goto=llm_analysis
2026-04-14 16:27:11,746 | INFO | [structure_analysis.llm_analysis] start
2026-04-14 16:27:11,747 | INFO | [structure_analysis.llm_analysis] dispatching boundary batch review (31 suspects)
2026-04-14 16:27:11,747 | INFO | [structure_analysis.llm_analysis] done goto=boundary_llm_batch
2026-04-14 16:27:11,752 | INFO | [structure_analysis.llm_analysis.boundary_batch] start
2026-04-14 16:27:17,653 | INFO | [structure_analysis.llm_analysis.boundary_batch] complete (31 reviews)
2026-04-14 16:27:17,654 | INFO | [structure_analysis.llm_analysis.boundary_batch] done go

In [4]:
for para in res.working_doc.paragraphs:
    if not para.text.strip():
        continue
    parser = para.meta.parser if para.meta else None
    if parser is None:
        continue
    print(f"{para.unit_id}: {parser.category} | clause: {parser.clause_no} | subclause: {parser.subclause_no} | {para.text[:120].replace("\n", " ")}")


s1.p1: appendix | clause: None | subclause: None | [별표1]
s1.p3: preamble | clause: None | subclause: None | 출판권 설정 계약서(제2조 제1호 관련)
s1.p4: input_block | clause: None | subclause: None | 저작권자1) _____(이하 ‘저작권자’라고 한다)와(과) 출판권자 _____(이하 ‘출판권자’라고 한다)는(은) 아래 대상 저작물에 대하여 아래와 같이 출판권설정계약을 체결한다.  대상 저작물의 표시 제호(가제) 
s1.p6: clause_heading | clause: 1 | subclause: None | 제1조 (계약의 목적)
s1.p7: clause_body | clause: 1 | subclause: None | 이 계약은 ‘저작권자’가 대상 저작물에 대하여 ‘출판권자’에게 출판권을 부여하고 ‘출판권자’는 이를 출판함에 있어 양 당사자 사이의 권리∙의무 및 필요한 제반 사항을 규정함을 목적으로 한다.
s1.p9: clause_heading | clause: 2 | subclause: None | 제2조 (정의)
s1.p10: clause_body | clause: 2 | subclause: None | 1. "대상 저작물"은 위에 표시한, 이 계약의 목적이 되는 저작물을 말한다.
s1.p11: clause_body | clause: 2 | subclause: None | 2. "복제"는 대상 저작물을 인쇄ㆍ사진촬영ㆍ복사ㆍ녹음ㆍ녹화 그 밖의 방법으로 일시적 또는 영구적으로 유형물에 고정하거나 다시 제작하는 것을 말한다.
s1.p12: clause_body | clause: 2 | subclause: None | 3. "배포"는 대상 저작물 원본 또는 그 복제물을 공중에게 대가를 받거나 받지 아니하고 양도 또는 대여하는 것을 말한다.
s1.p13: clause_body | clause: 2 | subclause: None | 4

Failed to export span batch due to timeout, max retries or shutdown.
